## Part 3 — Machine Learning: predict no-shows and choose the model honestly

### Task 3.1 — Features, split and baseline

Build X from age, distance_km, fee_inr, lead_days, prior_no_shows, reminder_sent (convert Yes/No to 1/0 with .map), and the one-hot encoded booking_channel and visit_type (pd.get_dummies(X, columns=["booking_channel","visit_type"])). Do not use patient_id, appointment_id, dates, clinic_id, gender or status as features. Target y = no_show. Split with train_test_split(X, y, test_size=0.2, random_state=42). Report the training and test sizes, the number of no-shows in the test set, and the baseline accuracy of always predicting "Attended". Do not shuffle or re-sort the rows after loading appointments_clean.csv: the exact test-set counts below depend on the row order of that file.

In [ ]:
#Step 1: Load the cleaned CSV
import pandas as pd
import numpy as np
df = pd.read_csv("appointments_clean.csv")

print(df.shape)
print(df.head())
print(df.columns)



(990, 16)
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show          1        1  


In [ ]:
#Step 2: Create X and y

X = df[['age', 'distance_km', 'fee_inr',
        'lead_days', 'prior_no_shows',
        'reminder_sent', 'booking_channel', 'visit_type']].copy()

#2. Convert reminder_sent into 1 and 0
X['reminder_sent'] = X['reminder_sent'].map({'Yes':1,'No':0})

#3. One-hot encode booking_channel and visit_type

X = pd.get_dummies(X,
                   columns = ['booking_channel','visit_type'],
                   dtype = int
                   )

#Create target variable y
y = df['no_show']

#Check your features
print(X.shape)
print(X.columns)
print(y.shape)

print(X['reminder_sent'].value_counts(dropna=False))
print("Missing values:", X.isna().sum().sum())
print("Shape:", X.shape)

(990, 11)
Index(['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows',
       'reminder_sent', 'booking_channel_App', 'booking_channel_Phone',
       'booking_channel_Walk-In', 'visit_type_Consultation',
       'visit_type_Follow-up'],
      dtype='object')
(990,)
reminder_sent
1    561
0    429
Name: count, dtype: int64
Missing values: 0
Shape: (990, 11)


In [ ]:
#Task 3.1 — Train/test split and baseline.

#Step 1: Split the data

from sklearn.model_selection import train_test_split

X_train,X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

print("Training Data:",X_train.shape)
print("Testing Data",X_test.shape)

Training Data: (792, 11)
Testing Data (198, 11)


In [ ]:
#Step 2: Count no-shows in the test set

print("No of Shows in the Test Set:",y_test.sum())

No of Shows in the Test Set: 49


In [ ]:
#Step 3: Calculate baseline accuracy
#Our baseline model will always predict Attended (0), without trying to identify no-shows.

baseline_accuracy = (y_test==0).mean()
print("Baseline Accuracy:",baseline_accuracy)

Baseline Accuracy: 0.7525252525252525


### Task 3.2 — Logistic regression

Scale with StandardScaler (scaler = StandardScaler(), X_train_s = scaler.fit_transform(X_train), X_test_s = scaler.transform(X_test) — fit on the training set only), fit LogisticRegression(max_iter=1000), and report accuracy, the confusion matrix, precision, recall, F1 and ROC-AUC on the test set. Print the coefficients (model.coef_[0], one per feature, in the column order of X) next to the feature names and report np.exp(coefficient) for reminder_sent. Remember that the model was fitted on scaled features: to get the effect of a reminder going from No (0) to Yes (1), divide the coefficient by the feature's standard deviation (scaler.scale_) before applying np.exp. State both numbers and explain in one sentence why they differ. Word your conclusion as an association ("in this model, holding the other features constant …"): the data are observational, so it does not prove that reminders cause fewer no-shows.

In [ ]:
# Step 1: Scale the features
# We will use StandardScaler

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# Learn scaling values from training data only
X_train_s = scaler.fit_transform(X_train)

# Apply the same scaling to test data
X_test_s = scaler.transform(X_test)

In [ ]:
#Step 2: Train Logistic Regression

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter = 1000)

lr_model.fit(X_train_s,y_train)

LogisticRegression(max_iter=1000)

In [ ]:
#Step 3: Make predictions
y_pred =lr_model.predict(X_test_s)
y_prob = lr_model.predict_proba(X_test_s)[:,1]

In [ ]:
#Step 4: Calculate evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 Score:", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

print("Confusion Matrix:",confusion_matrix(y_test, y_pred))

Accuracy: 0.788
Precision: 0.63
Recall: 0.347
F1 Score: 0.447
ROC-AUC: 0.789
Confusion Matrix: [[139  10]
 [ 32  17]]


In [ ]:
#Step 5: Interpret model coefficients
#which factors increase or decrease the likelihood of a patient missing an appointment.

coefficients = pd.DataFrame({
    'features':X.columns,
    'coefficients':lr_model.coef_[0]
}
)

coefficients['odds_ratio'] = np.exp(coefficients['coefficients'])

coefficients = coefficients.reindex(
    coefficients['coefficients'].abs().sort_values(
        ascending=False
    ).index
)

print(coefficients.round(3).reset_index(drop=True))





                   features  coefficients  odds_ratio
0            prior_no_shows         0.650       1.915
1                 lead_days         0.614       1.849
2             reminder_sent        -0.581       0.560
3                   fee_inr         0.453       1.573
4                       age        -0.254       0.776
5               distance_km         0.253       1.288
6     booking_channel_Phone         0.216       1.241
7   booking_channel_Walk-In        -0.186       0.830
8       booking_channel_App        -0.047       0.954
9   visit_type_Consultation         0.016       1.016
10     visit_type_Follow-up        -0.016       0.984


In [ ]:
#Step 6: Calculate the reminder effect

reminder_index = list(X.columns).index('reminder_sent')

reminder_coef = lr_model.coef_[0][reminder_index]

reminder_effect = np.exp(reminder_coef/scaler.scale_[reminder_index])

print("Reminder coefficient:", round(reminder_coef, 3))

print("Scaled odds ratio:", round(np.exp(reminder_coef), 3))

print("Actual No-to-Yes odds ratio:", round(reminder_effect, 3))


Reminder coefficient: -0.581
Scaled odds ratio: 0.56
Actual No-to-Yes odds ratio: 0.309


In this model, holding the other features constant, appointments with reminders are associated with approximately 69.1% lower predicted no-show odds compared with appointments without reminders.
\[
(1-0.309)\times100=69.1\%
\]

This is an association based on observational data and does not prove that reminders cause fewer no-shows.

### Task 3.3 — Decision tree and overfitting

Fit DecisionTreeClassifier(random_state=42) with no depth limit, and again with max_depth=3. For each, report training accuracy and test accuracy, and state in one sentence what the gap shows.

In [ ]:
#Step 1: Import Decision Tree
from sklearn.tree import DecisionTreeClassifier

#Step 2: Train the full-depth Decision Tree

dt_full = DecisionTreeClassifier(random_state = 42)

dt_full.fit(X_train,y_train)

train_accuracy_full = dt_full.score(X_train,y_train)
test_accuracy_full = dt_full.score(X_test,y_test)

print("Full Tree Training Accuracy:",round(train_accuracy_full, 3))

print("Full Tree Testing Accuracy:",round(test_accuracy_full, 3))

Full Tree Training Accuracy: 1.0
Full Tree Testing Accuracy: 0.672


In [ ]:
#Step 3: Train the restricted Decision Tree
dt_depth3 = DecisionTreeClassifier(max_depth = 3, random_state = 42)

dt_depth3.fit(X_train,y_train)

train_accuracy_depth3 = dt_depth3.score(X_train,y_train)
test_accuracy_depth3 = dt_depth3.score(X_test,y_test)

print("Depth3 Tree Training Accuracy:",round(train_accuracy_depth3, 3))

print("Depth3 Testing Accuracy:",round(test_accuracy_depth3, 3))

Depth3 Tree Training Accuracy: 0.822
Depth3 Testing Accuracy: 0.778


**conclusion**

The full-depth Decision Tree achieved 100% training accuracy but only 67.2% test accuracy, indicating significant overfitting. It learned the training data too closely and performed poorly on unseen data. Restricting the tree to a maximum depth of 3 reduced the gap between training and test accuracy, achieving 82.2% training accuracy and 77.8% test accuracy. Therefore, the depth-3 tree generalizes better than the full-depth tree.

### Task 3.4 — Random forest and a comparison table

Fit RandomForestClassifier(n_estimators=100, random_state=42). Report accuracy, recall and ROC-AUC (use predict_proba(X_test)[:, 1] for AUC) and the top three rf.feature_importances_ (the attribute name is given because the course mentioned the idea only briefly). Then build a table with one row each for logistic regression, the depth-3 tree and the random forest (accuracy, recall, ROC-AUC) and say which model looks best and why; this table is for reporting — the formal choice is made in Task 3.5.

In [ ]:
#step1 : Import Random Forest Classifier

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [ ]:
#Step 2: Calculate evaluation metrics

from sklearn.metrics import accuracy_score,recall_score,roc_auc_score

#Predict
rf_pred = rf_model.predict(X_test)

#No Show Probabilities
rf_proba = rf_model.predict_proba(X_test)[:, 1]

#Evaluation Metrics

print("Accuracy:",round(accuracy_score(y_test,rf_pred),3))

print("Recall:",round(recall_score(y_test,rf_pred),3))

print("ROC AUC Score:",round(roc_auc_score(y_test,rf_proba),3))




Accuracy: 0.747
Recall: 0.306
ROC AUC Score: 0.706


In [ ]:
#Step 3: Check training accuracy
rf_train_accuracy = rf_model.score(X_train,y_train)

print("Random Forest Training Accuracy:",
      round(rf_train_accuracy, 3))

Random Forest Training Accuracy: 1.0


In [ ]:
#step4 Find the top 3 important features
importances = pd.DataFrame({
    'Feature':X.columns,
    'Importance':rf_model.feature_importances_
}).sort_values('Importance',ascending = False)

print(importances.head(3).round(3))

       Feature  Importance
3    lead_days       0.220
0          age       0.197
1  distance_km       0.195


In [ ]:
#Step 5 : Calculate evaluation metrics for depth3 metrics

from sklearn.metrics import accuracy_score, recall_score,roc_auc_score

dt_depth3_pred = dt_depth3.predict(X_test)

dt_depth3_prob = dt_depth3.predict_proba(X_test)[:,1]

print("Accuracy:", round(accuracy_score(y_test, dt_depth3_pred), 3))
print("Recall:", round(recall_score(y_test, dt_depth3_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, dt_depth3_prob), 3))



Accuracy: 0.778
Recall: 0.224
ROC-AUC: 0.7


In [ ]:
#step 6 — Model comparison

comparison_table = pd.DataFrame({
    'Model':['Logistic Regression','DecisionTree(Depth3)','Random Forest'],
    'Accuracy':[0.788, 0.778, 0.747],
    'Recall':[0.347, 0.224, 0.306],
    'ROC-AUC':[0.789, 0.700, 0.706]
})

print(comparison_table)


                  Model  Accuracy  Recall  ROC-AUC
0   Logistic Regression     0.788   0.347    0.789
1  DecisionTree(Depth3)     0.778   0.224    0.700
2         Random Forest     0.747   0.306    0.706


Correct interpretation

- Logistic Regression: Best overall, with the highest accuracy, recall and ROC-AUC.
- Decision Tree: Lower recall and ROC-AUC.
- Random Forest: Better ROC-AUC than the depth-3 tree,      but lower accuracy and recall than Logistic Regression.

Assignment conclusion:

Logistic Regression performs best among the three models, achieving the highest test accuracy (0.788), recall (0.347), and ROC-AUC (0.789). Therefore, it appears to be the strongest model based on the test-set reporting results. However, final model selection will be based on cross-validation using the training data only, as required by the project.

### Task 3.5 — Tune, choose the model and the threshold, then test once

(a) Run GridSearchCV(RandomForestClassifier(random_state=42), {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}, cv=5, scoring="roc_auc") on the training set; print best_params_ and best_score_ (a cross-validated ROC-AUC).

(b) Choose the final model without touching the test set: compute the 5-fold cross-validated ROC-AUC on the training set for logistic regression, the depth-3 tree and the random forest — cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc").mean() — and compare them with the tuned forest's best_score_. For logistic regression, give cross_val_score the unscaled X_train together with the supplied pipeline below, so that the scaler is re-fitted on each fold's own training part (scaling the whole training set first would let each validation fold influence the scaler). The highest wins.

(c) Choose the threshold without touching the test set: get out-of-fold training probabilities for the winner with cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1] (for logistic regression use the pipeline and the unscaled X_train) and compare thresholds 0.5 and 0.35 on precision, recall, F1 and number flagged; pick the threshold with the higher F1 and say why recall matters here (a missed warning costs an empty slot, an extra call costs a minute).

 (d) Fit the winner on the training set (the pipeline fits its own scaler) and evaluate it once on the test set at your chosen threshold: precision, recall, number flagged and ROC-AUC.

 (e) Save test_risk_scores.csv with columns appointment_id, no_show, risk_prob, flag (1 if risk_prob ≥ your threshold) for every test row. (To get appointment_id, keep it aside before building X, or use X_test.index to look it up in the loaded DataFrame.) In one sentence, say how a clinic would really choose the threshold: by how many follow-up calls the front desk can make in an evening, against the cost of an empty slot.

Supplied pipeline for logistic regression (a pipeline chains the same fit/transform steps you used in 3.2 and repeats them inside every fold; the idea is the one from 20.2, the code is given because it was not shown in class):

#### step1  — Grid Search with 5-fold Cross-Validation
(a) Run GridSearchCV(RandomForestClassifier(random_state=42), {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}, cv=5, scoring="roc_auc") on the training set; print best_params_ and best_score_ (a cross-validated ROC-AUC).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid ={
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
    }

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state = 42),
    param_grid,
    cv =5,
    scoring ="roc_auc"
)

rf_grid.fit(X_train,y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best CV ROC-AUC:", round(rf_grid.best_score_,3))





Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.767


In [ ]:
# Step 2 — Compare candidate models using 5-fold cross-validation

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

lr_pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('model',LogisticRegression(max_iter = 1000))

])

In [ ]:
# Calculate CV ROC-AUC scores

#Logistic Regression
lr_cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv = 5, scoring = 'roc_auc').mean()

## Decision Tree (Depth 3)

dt_depth3_cv_auc = cross_val_score(dt_depth3,X_train,y_train, cv = 5,scoring = 'roc_auc').mean()

#Original Random Forest
rf_cv_auc = cross_val_score(rf_model,X_train,y_train, cv = 5,scoring = 'roc_auc').mean()

#Tuned Random Forest
tuned_rf_cv_auc = rf_grid.best_score_

print("Logistic Regression CV ROC-AUC:", round(lr_cv_auc, 3))
print("Decision Tree CV ROC-AUC:", round(dt_depth3_cv_auc, 3))
print("Random Forest CV ROC-AUC:", round(rf_cv_auc, 3))
print("Tuned Random Forest CV ROC-AUC:", round(tuned_rf_cv_auc, 3))



Logistic Regression CV ROC-AUC: 0.774
Decision Tree CV ROC-AUC: 0.678
Random Forest CV ROC-AUC: 0.747
Tuned Random Forest CV ROC-AUC: 0.767


**Selected model**
            
            Logistic Regression

Why are we selecting Logistic Regression?

- It achieved the highest CV ROC-AUC of 0.774.
- It performed slightly better than the tuned Random Forest (0.767).
- It generalizes better according to these cross-validation results.

conclusion:

Logistic Regression was selected as the final candidate model because it achieved the highest mean 5-fold cross-validation ROC-AUC of 0.774. Although the tuned Random Forest performed competitively with a score of 0.767, Logistic Regression showed the best overall cross-validation performance.

In [ ]:
#Step 3 — Generate Out-of-Fold Predictions

from sklearn.model_selection import cross_val_predict
oof_prob = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

print("OOF probabilities:", oof_prob[:10])
print("Number of predictions:", len(oof_prob))

OOF probabilities: [0.44277273 0.11344644 0.34663886 0.12072607 0.17356387 0.24988488
 0.12508639 0.4103534  0.16669732 0.08633667]
Number of predictions: 792


In [ ]:
#Compare thresholds 0.50 and 0.35
from sklearn.metrics import precision_score,recall_score,f1_score

for thresholds in [0.50,0.35]:
  oof_pred = (oof_prob>thresholds).astype(int)

  precision = precision_score(y_train,oof_pred)
  recall = recall_score(y_train,oof_pred)
  f1 = f1_score(y_train,oof_pred)

  flagged = oof_pred.sum()

  print("\nThreshold:", thresholds)
  print("Precision:", round(precision, 3))
  print("Recall:", round(recall, 3))
  print("F1 Score:", round(f1, 3))
  print("Flagged appointments:", flagged)


Threshold: 0.5
Precision: 0.575
Recall: 0.249
F1 Score: 0.347
Flagged appointments: 73

Threshold: 0.35
Precision: 0.513
Recall: 0.467
F1 Score: 0.489
Flagged appointments: 154


conclusion

A threshold of 0.35 was selected instead of the default 0.50 because it achieved a higher recall (0.467) and F1-score (0.489). Although precision decreased from 0.575 to 0.513, the lower threshold identified a larger proportion of potential no-shows. This is suitable for the CareLoop project, where identifying more at-risk appointments is important for reducing missed appointments.

Now we have selected:
- Model: Logistic Regression
- Threshold: 0.35
We will train the selected pipeline on the complete training dataset and then evaluate it on the test dataset.

In [ ]:
#Step 4 — Fit the final model
# Train final model using all training data
lr_pipe.fit(X_train,y_train)

#Predict no show probabilities on test data
test_prob =lr_pipe.predict_proba(X_test)[:,1]

# Apply selected threshold
test_pred = (test_prob > 0.35).astype(int)

print("Final model trained successfully!")
print("Selected threshold: 0.35")
print("Test predictions:", len(test_pred))

Final model trained successfully!
Selected threshold: 0.35
Test predictions: 198


In [ ]:
#Step 6 — Final test evaluation

print("Final Test Results")
print("-------------------")

print("Accuracy:", round(accuracy_score(y_test, test_pred), 3))
print("Precision:", round(precision_score(y_test, test_pred), 3))
print("Recall:", round(recall_score(y_test, test_pred), 3))
print("F1 Score:", round(f1_score(y_test, test_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, test_prob), 3))

print("Flagged appointments:", test_pred.sum())

Final Test Results
-------------------
Accuracy: 0.788
Precision: 0.571
Recall: 0.571
F1 Score: 0.571
ROC-AUC: 0.789
Flagged appointments: 49


In [ ]:
#Step 7 — Save test risk scores to CSV

risk_scores = pd.DataFrame({
    'appointment_id':df.loc[X_test.index,'appointment_id'],
    'no_show':y_test,
    'risk_prob':np.round(test_prob,2),
    'flag':test_pred
})

# Save as CSV
risk_scores.to_csv('test_risk_scores.csv', index=False)

print("File saved successfully!")
print("Shape:", risk_scores.shape)

# Display first 5 records
print(risk_scores.head())

File saved successfully!
Shape: (198, 4)
    appointment_id  no_show  risk_prob  flag
213          A0217        0       0.02     0
331          A0336        0       0.15     0
501          A0508        0       0.52     1
309          A0314        0       0.19     0
88           A0089        0       0.17     0


In [ ]:
final_check = pd.read_csv('test_risk_scores.csv')

print("Shape:", final_check.shape)
print("Columns:", final_check.columns.tolist())
print("Flagged appointments:", final_check['flag'].sum())
print("No-show records:", final_check['no_show'].sum())

Shape: (198, 4)
Columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']
Flagged appointments: 49
No-show records: 49
